<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Applied_Structured_Outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applied Structured Outputs: Three Mini Projects

The previous lesson taught the *technique* — schema in, validated object out. This lesson is about what turns a technique into a **production system**: the three or four decisions around the schema that decide whether your pipeline is safe to leave running unattended.

Three small, production-flavored projects, each built around one such decision:

1. **Ticket triage** — schema design as routing policy, plus **confidence gating**: the model handles the easy 80%, humans get the ambiguous 20%.
2. **Log analysis** — **redact secrets before the LLM sees anything**, then distill the noise into a structured incident report.
3. **Bulk enrichment** — the same extraction at scale: a sequential loop, then **capped async concurrency**, then the **Batch API at half price**.

Every project runs on the `extract()` helper from the previous lesson — same code on Gemini (course default), OpenAI, or Anthropic.

## 🧭 What You'll Learn

- Designing schemas that encode **business decisions** (`Literal` categories as routing lanes, priorities as SLAs) — not just data shapes
- **Confidence gating**: asking the model to score its own certainty, and routing low-confidence cases to human review instead of acting on them
- **Redaction before inference**: why raw logs never go to a third-party API, and a regex pass that strips keys, tokens, and emails first
- Scaling one extraction to many: sequential → `asyncio` with a **semaphore cap** → provider **Batch APIs** at ~50% price (as of July 2026)
- The quiet superpower of typed outputs: every project ends in objects you can queue, file, or load into a table — no parsing code anywhere

## 1. Setup: Environment, Keys, and Provider

The standard course setup cell — chat only, one provider key (whichever `PROVIDER` you select; Gemini remains the course default, and the optional Batch experiment at the end uses Gemini's Batch API).

The **model field is an editable dropdown** (`{allow-input: true}`): pick one of the listed course defaults, or type any newer model ID straight into the box — no code changes needed. (Locally, simply edit the string.) All three projects are high-volume, low-difficulty work, which is what `gemini-3.5-flash-lite` is built for: Google positions it as the high-throughput, low-cost choice for classification and extraction, and it is one of the options in the dropdown.

In [1]:
# ============================================================
# ⚙️ Setup — environment, dependencies, API keys, provider
# ============================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Pick your model providers (dropdowns in Colab; edit the values locally)
PROVIDER = "gemini"  # @param ["gemini", "openai", "anthropic"]

# Pick a model for the selected provider — or TYPE any newer model ID into the
# box (the dropdown is editable thanks to allow-input):
CHAT_MODEL = "gemini-3.7-flash"  # @param ["gemini-3.7-flash", "gemini-3.5-flash-lite", "gpt-5.6-luna", "claude-sonnet-5"] {allow-input: true}

_KEY_FOR = {"gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}
REQUIRED_KEYS = sorted({_KEY_FOR[PROVIDER]})

if IN_COLAB:
    import importlib
    import site
    import subprocess

    # Shared install profile, pinned course-wide (July 2026).
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "google-genai==2.3.0",
            "openai==2.46.0",
            "anthropic==0.117.0",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without a runtime restart

    # In Colab: Secrets tab (🔑 icon) → Add new secret → e.g. GOOGLE_API_KEY
    from google.colab import userdata

    for key in REQUIRED_KEYS:
        os.environ[key] = userdata.get(key)

if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

    # Locally: dependencies are installed once from the repo's requirements.
    # Keys live in a .env file at the repo root.
    from dotenv import load_dotenv

    load_dotenv()
    missing = [k for k in REQUIRED_KEYS if not os.getenv(k)]
    assert not missing, f"Missing from .env: {missing}"

print(f"✅ Setup complete — {'Colab' if IN_COLAB else 'local'} | chat: {PROVIDER}")

✅ Setup complete — local | chat: gemini


## 2. The Helpers: `generate()` and `extract()`

📎 *Both unchanged — `generate()` from "How To Use LLMs via API", `extract()` from the structured-outputs lesson. This whole lesson is `extract()` doing honest work.*

In [2]:
# 📎 generate() and embed() are the course's two provider helpers,
#    built in “How To Use LLMs via API” and “Basic RAG” — unchanged here.
from anthropic import Anthropic
from google import genai
from google.genai import types as genai_types
from openai import OpenAI

# Course-standard default models per provider (August 2026)
MODELS = {
    "gemini": "gemini-3.7-flash",
    "openai": "gpt-5.6-luna",
    "anthropic": "claude-sonnet-5",
}

# The setup-cell form selection (or any typed model ID) overrides the default:
MODELS[PROVIDER] = CHAT_MODEL

# Create only the clients we actually need
if PROVIDER == "gemini":
    gemini_client = genai.Client()
if PROVIDER == "openai":
    openai_client = OpenAI()
if PROVIDER == "anthropic":
    anthropic_client = Anthropic()


def generate(prompt, system=None, model=None):
    """Send one prompt to the selected PROVIDER and return the reply text."""
    if PROVIDER == "gemini":
        response = gemini_client.models.generate_content(
            model=model or MODELS["gemini"],
            contents=prompt,
            config=genai_types.GenerateContentConfig(system_instruction=system),
        )
        return response.text

    if PROVIDER == "openai":
        response = openai_client.responses.create(
            model=model or MODELS["openai"],
            instructions=system,
            input=prompt,
            reasoning={"effort": "none"},
        )
        return response.output_text

    if PROVIDER == "anthropic":
        response = anthropic_client.messages.create(
            model=model or MODELS["anthropic"],
            max_tokens=4096,
            **({"system": system} if system else {}),
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


In [3]:
# 📎 extract() from the structured-outputs lesson — unchanged.
#    Prompt in, validated Pydantic instance out, on the selected PROVIDER.
def extract(prompt, schema, system=None):
    """Native structured output on the selected PROVIDER.

    Returns a validated instance of `schema` (a Pydantic BaseModel subclass).
    """
    if PROVIDER == "gemini":
        response = gemini_client.models.generate_content(
            model=MODELS["gemini"],
            contents=prompt,
            config=genai_types.GenerateContentConfig(
                system_instruction=system,
                response_mime_type="application/json",
                response_schema=schema,
            ),
        )
        return response.parsed or schema.model_validate_json(response.text)

    if PROVIDER == "openai":
        response = openai_client.responses.parse(
            model=MODELS["openai"],
            instructions=system,
            input=prompt,
            text_format=schema,
            reasoning={"effort": "none"},
        )
        return response.output_parsed

    if PROVIDER == "anthropic":
        tool = {
            "name": "record_result",
            "description": "Record the structured result of the task.",
            "input_schema": schema.model_json_schema(),
        }
        response = anthropic_client.messages.create(
            model=MODELS["anthropic"],
            max_tokens=4096,
            **({"system": system} if system else {}),
            tools=[tool],
            tool_choice={"type": "tool", "name": "record_result"},
            messages=[{"role": "user", "content": prompt}],
        )
        block = next(b for b in response.content if b.type == "tool_use")
        return schema.model_validate(block.input)

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")

## 3. Project 1 — Ticket Triage with Confidence Gating

The AI Tutor has a support inbox. Most tickets are routine ("where are the slides?"), some are radioactive ("40 people blocked"). The goal is not "classify tickets" — it is **route work safely**: let the model draft and file the easy ones, and *escalate its own uncertainty* instead of guessing.

Two design moves carry the whole project:

- The schema's `Literal` fields are **policy**, not decoration: five categories because there are five queues; four priorities because there are four response-time SLAs.
- A `confidence` field turns the classifier into a **gate**. Models are miscalibrated optimists, so the threshold is something you *tune on labeled tickets* — but even a rough gate converts "fully automated and occasionally wrong" into "mostly automated and reviewable".

In [4]:
from typing import Literal

from pydantic import BaseModel, Field


class TicketTriage(BaseModel):
    """Routing decision for one support ticket."""

    category: Literal["account", "billing", "course_content", "technical", "other"] = Field(
        description="The single best routing queue for this ticket."
    )
    priority: Literal["low", "medium", "high", "urgent"] = Field(
        description="Business urgency (impact and blockage), not customer tone."
    )
    summary: str = Field(description="The issue in one sentence, at most 15 words.")
    reply_draft: str = Field(description="A short, polite first reply (2-3 sentences), ready to send.")
    confidence: float = Field(
        description=(
            "Your confidence, between 0.0 and 1.0, that `category` is correct. "
            "Go BELOW 0.8 whenever the ticket is ambiguous, mixes several intents, "
            "or does not fit any category cleanly."
        )
    )


TICKETS = [
    "I bought the course yesterday but the payment went through twice on my card. Please refund one of the charges.",
    "The Colab notebook in the vector-database lesson crashes at the setup cell with ModuleNotFoundError: chromadb.",
    "Where can I find the slides for the RAG evaluation lesson? The video mentions a download link but I don't see it.",
    "I can't log in since changing my email address, and password-reset emails never arrive.",
    "Your AI tutor told me to delete my .env file and now my API keys are gone?? This cost me a whole afternoon.",
    "Do you offer student discounts? Also, is there a certificate at the end, and does it expire?",
    "The heading-aware chunking lesson says 800-token chunks, but an earlier notebook used 512 — which one should I use for my project?",
    "URGENT: our whole team shows 'subscription expired' but our company renewed last week. 40 people are blocked.",
]

print(f"{len(TICKETS)} tickets in the inbox.")

8 tickets in the inbox.


In [5]:
TRIAGE_PROMPT = """You triage support tickets for an online AI-engineering course
(the product includes course content, notebooks, an AI tutor, accounts, and billing).

Triage the ticket below.

<ticket>
{ticket}
</ticket>"""

PRIORITY_ICON = {"urgent": "🔴", "high": "🟠", "medium": "🟡", "low": "🟢"}

triaged = []
for ticket in TICKETS:
    decision = extract(TRIAGE_PROMPT.format(ticket=ticket), TicketTriage)
    triaged.append({"ticket": ticket, "decision": decision})
    d = decision
    print(f"{PRIORITY_ICON[d.priority]} {d.category:<15} conf={d.confidence:.2f}  {d.summary}")

🟠 billing         conf=0.98  Customer was charged twice for the course purchase and requests a refund.


🟡 course_content  conf=0.85  The vector-database lesson Colab notebook fails with a missing chromadb module error.


🟢 course_content  conf=0.95  User cannot find the download link for the RAG evaluation lesson slides.


🟠 account         conf=0.95  User cannot log in or receive password-reset emails after changing their email address.


🟡 technical       conf=0.85  AI tutor advised the user to delete their .env file, losing API keys.


🟢 billing         conf=0.65  Inquiry about student discount availability and certificate expiration details.


🟢 course_content  conf=0.95  Learner asks whether to use 800 or 512-token chunks for their project.


🔴 billing         conf=0.95  A team of 40 users is blocked due to an inaccurate subscription expiration error.


In [6]:
CONFIDENCE_GATE = 0.80  # tune this on a labeled sample — do not ship a vibe

auto_queue = [t for t in triaged if t["decision"].confidence >= CONFIDENCE_GATE]
review_queue = [t for t in triaged if t["decision"].confidence < CONFIDENCE_GATE]

print(f"🤖 auto-handled: {len(auto_queue)}   🧑 human review: {len(review_queue)}\n")

print("── human review queue " + "─" * 40)
for t in review_queue:
    d = t["decision"]
    print(f"  {PRIORITY_ICON[d.priority]} conf={d.confidence:.2f} [{d.category}] {t['ticket'][:70]}…")

print("\n── sample auto-reply (highest-confidence ticket) " + "─" * 14)
best = max(auto_queue, key=lambda t: t["decision"].confidence)
print(f"  ticket: {best['ticket'][:70]}…")
print(f"  draft:  {best['decision'].reply_draft}")

🤖 auto-handled: 7   🧑 human review: 1

── human review queue ────────────────────────────────────────
  🟢 conf=0.65 [billing] Do you offer student discounts? Also, is there a certificate at the en…

── sample auto-reply (highest-confidence ticket) ──────────────
  ticket: I bought the course yesterday but the payment went through twice on my…
  draft:  Thank you for bringing this to our attention, and we apologize for the duplicate charge. Our billing team is looking into your transaction and will process the refund for the extra charge shortly. We will follow up as soon as the refund has been completed.


**What just happened?** Every ticket came back as a typed `TicketTriage` object — category constrained to the five real queues (a `Literal` makes "Billing-ish?" impossible), a priority, a ready-to-send draft, and a self-reported confidence. The gate then did the production move: **the cost of a wrong automated action is asymmetric** (a bad auto-reply to the 40-people-blocked ticket is a fire; a human skimming an easy ticket is a shrug), so low confidence routes to people. Expect the ambiguous tickets — the multi-intent ones, the tutor-deleted-my-env one — in the review queue (**your split may differ**; that variability is *why* the gate exists).

## 4. Project 2 — Log Analysis with Secret Redaction

Logs are where incidents get diagnosed — and where credentials go to leak. The moment you pipe logs to a third-party API, you must assume every byte is stored somewhere you don't control. So the iron rule of this project: **redact locally, before the request leaves the machine.** Cheap regexes first, the expensive model second.

The log below is a synthetic (and clearly fake-keyed) excerpt from a bad night for the tutor's answer service.

In [7]:
RAW_LOG = """2026-07-25 03:12:04 UTC api-gw   INFO  GET /v1/answer user=priya@example.com q="what is RRF"
2026-07-25 03:12:09 UTC chroma   WARN  query latency 3400ms (p95 budget 800ms) collection=ai_tutor_knowledge
2026-07-25 03:12:11 UTC worker   ERROR EmbedTimeout retry 1/3 provider=gemini key=AIzaSyEXAMPLE-not-a-real-key-0000000
2026-07-25 03:12:14 UTC worker   ERROR EmbedTimeout retry 2/3 provider=gemini key=AIzaSyEXAMPLE-not-a-real-key-0000000
2026-07-25 03:12:19 UTC auth     INFO  service token issued: Bearer eyJhbGciOiJIUzI1NiEXAMPLE.not-real.token000
2026-07-25 03:12:22 UTC api-gw   ERROR 502 from answer-svc user=dave@example.org
2026-07-25 03:12:31 UTC api-gw   ERROR 502 from answer-svc user=lin@example.net
2026-07-25 03:12:40 UTC worker   ERROR EmbedTimeout retry 3/3 provider=gemini key=AIzaSyEXAMPLE-not-a-real-key-0000000
2026-07-25 03:12:41 UTC worker   FATAL embedding provider unreachable, marking answer-svc unhealthy
2026-07-25 03:13:02 UTC lb       WARN  answer-svc removed from rotation (0/2 healthy)
2026-07-25 03:13:30 UTC oncall   INFO  paged: answer-svc down, error budget burning
2026-07-25 03:14:41 UTC deploy   INFO  rollback answer-svc to build 2026.07.19 initiated by oncall
2026-07-25 03:15:03 UTC api-gw   INFO  p95 latency 640ms, error rate back under 0.1%"""

print(RAW_LOG[:300], "…")

2026-07-25 03:12:04 UTC api-gw   INFO  GET /v1/answer user=priya@example.com q="what is RRF"
2026-07-25 03:12:09 UTC chroma   WARN  query latency 3400ms (p95 budget 800ms) collection=ai_tutor_knowledge
2026-07-25 03:12:11 UTC worker   ERROR EmbedTimeout retry 1/3 provider=gemini key=AIzaSyEXAMPLE-no …


In [8]:
import re

# Order matters: redact the most specific patterns first, emails last.
REDACTIONS = [
    (re.compile(r"AIza[0-9A-Za-z_\-]{10,}"), "<REDACTED_GOOGLE_KEY>"),
    (re.compile(r"sk-[0-9A-Za-z_\-]{10,}"), "<REDACTED_OPENAI_KEY>"),
    (re.compile(r"Bearer\s+[0-9A-Za-z_\-\.]+"), "Bearer <REDACTED_TOKEN>"),
    (re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"), "<REDACTED_EMAIL>"),
]


def redact(text):
    """Strip credential-shaped strings and emails BEFORE any API call."""
    n_total = 0
    for pattern, replacement in REDACTIONS:
        text, n = pattern.subn(replacement, text)
        n_total += n
    return text, n_total


safe_log, n_redacted = redact(RAW_LOG)
print(f"{n_redacted} sensitive strings redacted.\n")
print("\n".join(line for line in safe_log.splitlines() if "REDACTED" in line))

7 sensitive strings redacted.

2026-07-25 03:12:04 UTC api-gw   INFO  GET /v1/answer user=<REDACTED_EMAIL> q="what is RRF"
2026-07-25 03:12:11 UTC worker   ERROR EmbedTimeout retry 1/3 provider=gemini key=<REDACTED_GOOGLE_KEY>
2026-07-25 03:12:14 UTC worker   ERROR EmbedTimeout retry 2/3 provider=gemini key=<REDACTED_GOOGLE_KEY>
2026-07-25 03:12:19 UTC auth     INFO  service token issued: Bearer <REDACTED_TOKEN>
2026-07-25 03:12:22 UTC api-gw   ERROR 502 from answer-svc user=<REDACTED_EMAIL>
2026-07-25 03:12:31 UTC api-gw   ERROR 502 from answer-svc user=<REDACTED_EMAIL>
2026-07-25 03:12:40 UTC worker   ERROR EmbedTimeout retry 3/3 provider=gemini key=<REDACTED_GOOGLE_KEY>


In [9]:
class IncidentReport(BaseModel):
    """A structured incident summary built strictly from log evidence."""

    severity: Literal["sev1", "sev2", "sev3"] = Field(
        description="sev1 = user-facing outage, sev2 = degraded service, sev3 = minor issue."
    )
    affected_components: list[str] = Field(description="Component names exactly as they appear in the log.")
    summary: str = Field(description="What happened, in at most 2 sentences.")
    timeline: list[str] = Field(description="Key events in order, each formatted 'HH:MM:SS — event'.")
    probable_cause: str = Field(description="Most likely root cause, stated as a hypothesis.")
    next_steps: list[str] = Field(description="2-4 concrete follow-up actions.")
    unredacted_secrets_spotted: bool = Field(
        description="true if any credential-like string appears UNredacted in the provided log."
    )


INCIDENT_PROMPT = """Analyze this (already-redacted) production log excerpt and write the incident report.
Use only what the log supports; mark speculation as such in probable_cause.

<log>
{log}
</log>"""

report = extract(INCIDENT_PROMPT.format(log=safe_log), IncidentReport)

print(f"severity: {report.severity} | components: {', '.join(report.affected_components)}")
print(f"\nsummary: {report.summary}")
print(f"\nprobable cause: {report.probable_cause}")
print("\ntimeline:")
for event in report.timeline:
    print(f"  • {event}")
print("\nnext steps:")
for step in report.next_steps:
    print(f"  → {step}")
print(f"\nunredacted secrets spotted by the model: {report.unredacted_secrets_spotted}")

severity: sev1 | components: api-gw, chroma, worker, auth, lb, oncall, deploy, answer-svc

summary: The answer-svc service became unhealthy and was removed from rotation after repeated embedding timeouts to the Gemini provider, causing 502 errors for end users. The outage was mitigated after oncall rolled back answer-svc to build 2026.07.19.

probable cause: Hypothesis: The latest answer-svc build introduced a regression or misconfiguration in external Gemini embedding calls and timeout handling, causing complete instance health failure instead of graceful degradation.

timeline:
  • 03:12:04 — Incoming request received by api-gw
  • 03:12:09 — chroma reported high query latency of 3400ms
  • 03:12:11 — worker encountered first EmbedTimeout against gemini
  • 03:12:14 — worker encountered second EmbedTimeout retry
  • 03:12:19 — auth issued service token
  • 03:12:22 — api-gw reported 502 error from answer-svc
  • 03:12:31 — api-gw reported another 502 error from answer-svc
  • 03:12:4

**What just happened?** Fourteen lines of 3 a.m. noise became a filed-and-typed `IncidentReport` — severity from a closed set, a timeline your status page can render, next steps your ticket system can ingest. And the security posture is **defense in depth**: regexes stripped every key, token, and email *before* the request left the machine (deterministic, auditable, free), and the schema's `unredacted_secrets_spotted` field asks the model to act as a second pair of eyes on whatever survived. The model is the *last* line, never the first — a probabilistic redactor guarding real credentials is how leaks happen.

## 5. Project 3 — Bulk Enrichment: Sequential → Capped Async → Batch

One extraction is a demo; production is *ten thousand of them* — enriching every record in a catalog, every chunk in a knowledge base (📎 exactly what the production tutor does when it generates contextual summaries at ingestion). The pattern has three gears, and knowing when to shift is the skill:

1. **Sequential** — fine for dozens; painful past that (latency adds up linearly).
2. **Async with a concurrency cap** — the workhorse: N requests in flight, bounded by a semaphore so you respect rate limits instead of discovering them as `429`s.
3. **Batch API** — for jobs that can wait: submit everything, collect within 24 hours, pay **half price**.

Our catalog: the course glossary.

In [10]:
TERMS = [
    "chunking", "embedding", "cosine similarity", "vector database",
    "BM25", "Reciprocal Rank Fusion", "reranking", "hit rate",
    "MRR", "prompt injection", "hybrid search", "context window",
]


class GlossaryCard(BaseModel):
    """One enriched glossary entry for the course."""

    term: str
    definition: str = Field(description="At most 25 words, plain language, no hype.")
    category: Literal["retrieval", "generation", "evaluation", "security", "infrastructure"]
    related_terms: list[str] = Field(description="2-3 other terms from THIS glossary it connects to.")


CARD_PROMPT = """Write the glossary card for the term below, as used in a RAG / LLM-engineering course.
The glossary contains these terms (use them for related_terms): {all_terms}

Term: {term}"""

import time

t0 = time.perf_counter()
sequential_cards = [
    extract(CARD_PROMPT.format(term=term, all_terms=", ".join(TERMS)), GlossaryCard)
    for term in TERMS[:4]  # just a taste — enough to measure the pace
]
sequential_pace = (time.perf_counter() - t0) / len(sequential_cards)

print(f"Sequential: {sequential_pace:.1f}s per card → "
      f"~{sequential_pace * len(TERMS):.0f}s projected for all {len(TERMS)} terms")
print(f"\n{sequential_cards[0].term}: {sequential_cards[0].definition}")

Sequential: 2.4s per card → ~29s projected for all 12 terms

chunking: The process of splitting large documents into smaller, manageable text segments for embedding, indexing, and retrieval.


In [11]:
import asyncio

MAX_CONCURRENT = 4  # the cap is the point: stay under rate limits ON PURPOSE


async def enrich(term, semaphore):
    async with semaphore:  # at most MAX_CONCURRENT calls in flight
        return await asyncio.to_thread(
            extract, CARD_PROMPT.format(term=term, all_terms=", ".join(TERMS)), GlossaryCard
        )


async def enrich_all(terms):
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    return await asyncio.gather(*(enrich(term, semaphore) for term in terms))


t0 = time.perf_counter()
cards = await enrich_all(TERMS)  # top-level await works in Colab and Jupyter
async_seconds = time.perf_counter() - t0

print(f"Async with cap={MAX_CONCURRENT}: {len(cards)} cards in {async_seconds:.0f}s "
      f"(vs ~{sequential_pace * len(TERMS):.0f}s sequential)\n")
for card in cards[:5]:
    print(f"  [{card.category:<14}] {card.term}: {card.definition}")

Async with cap=4: 12 cards in 8s (vs ~29s sequential)

  [retrieval     ] chunking: The process of splitting large texts into smaller segments to make them easier to embed, index, search, and fit into context limits.
  [retrieval     ] embedding: A numerical vector representation of text that captures semantic meaning for mathematical comparison and search.
  [retrieval     ] cosine similarity: A mathematical metric measuring the angle between two vectors to determine their directional alignment and semantic closeness.
  [infrastructure] vector database: A specialized storage system designed to index, store, and quickly search high-dimensional vector embeddings based on similarity.
  [retrieval     ] BM25: A keyword-based ranking algorithm that scores document relevance based on term frequency, inverse document frequency, and document length.


**What just happened?** Same `extract()`, same schema — the only change was *scheduling*. With four requests in flight the wall clock dropped to roughly a quarter of sequential (**your speedup will vary** with provider latency and rate limits). The semaphore is not a performance trick, it is a *politeness contract*: unbounded `gather` over ten thousand items is a self-inflicted denial-of-service that ends in `429`s, retry storms, and a spiky bill. Cap first, then scale the cap.

For truly large jobs, stop paying interactive prices at all — that's the third gear.

## 6. The Third Gear: Batch APIs at Half Price

All three course providers run a batch lane: **submit a file or list of requests, get results within 24 hours (usually much faster), pay ~50% of interactive price** — Gemini's Batch API, OpenAI's Batch API, and Anthropic's Message Batches all follow the same *submit → poll → collect* shape (pricing as of July 2026). Structured outputs ride along: each batched request can carry the same `response_schema` you used interactively.

When batch is the right gear: nightly catalog enrichment, backfilling a new field over every record you own, re-running an eval suite, re-summarizing a knowledge base after a chunker change. When it is not: anything a user is waiting on.

The optional cell below submits our glossary job to **Gemini's** Batch API (the course default provider) and polls briefly — a tiny job usually clears in minutes, but the *contract* is only "within 24h", so the cell also shows how to come back for results later.

In [12]:
# 🔬 OPTIONAL EXPERIMENT — the same enrichment as a Gemini Batch job at ~50% price
if PROVIDER == "gemini":
    batch_requests = [
        {
            "contents": [{"role": "user", "parts": [{"text": CARD_PROMPT.format(term=term, all_terms=", ".join(TERMS))}]}],
            "config": {
                "response_mime_type": "application/json",
                "response_schema": GlossaryCard,  # structured outputs work in batch too
            },
        }
        for term in TERMS
    ]

    batch_job = gemini_client.batches.create(
        model=f"models/{MODELS['gemini']}",
        src=batch_requests,
        config={"display_name": "glossary-enrichment"},
    )
    print("Submitted:", batch_job.name)

    DEADLINE_S = 360  # poll here for up to 6 minutes; the API's own promise is 24h
    t0 = time.time()
    while time.time() - t0 < DEADLINE_S:
        batch_job = gemini_client.batches.get(name=batch_job.name)
        if batch_job.state.name in ("JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED"):
            break
        time.sleep(15)

    print("State:", batch_job.state.name)
    if batch_job.state.name == "JOB_STATE_SUCCEEDED":
        for item in batch_job.dest.inlined_responses[:3]:
            print("  •", (item.response.text or "")[:110].replace("\n", " "), "…")
    else:
        print("Not finished yet — perfectly normal. Collect later with:")
        print(f'  gemini_client.batches.get(name="{batch_job.name}")')
else:
    print("The batch demo targets the Gemini Batch API — set PROVIDER = 'gemini' to run it.")
    print("(OpenAI's Batch API and Anthropic's Message Batches use the same submit→poll→collect shape.)")

Submitted: batches/mqu7c0jzyfdd9kdobwiy4dhj91dpcis5qbg6


State: JOB_STATE_SUCCEEDED
  • {"term":"chunking","definition":"The process of breaking large text documents into smaller, coherent segments  …
  • {"term":"embedding","definition":"A numerical vector representation of text that captures semantic meaning, al …
  • {"term":"cosine similarity","definition":"A metric measuring the angle between two vectors to evaluate their s …


**What just happened?** The identical twelve requests went into a queue instead of a socket, tagged with the same `GlossaryCard` schema — and will cost about half as much. That is the entire trade: **latency for price**. In a real pipeline the polling loop lives in a scheduler, not a notebook cell, and the job id (`batch_job.name`) is the thing you persist.

One habit to keep from all three projects: the deliverable was never prose — it was a *typed object* (`TicketTriage`, `IncidentReport`, `GlossaryCard`) that the next system ingests without a parser. Design the schema first; the prompt is just how you fill it.

## 🔑 Key Takeaways

- Schemas encode **policy**: `Literal` categories are routing lanes, priorities are SLAs — design the type so invalid business states are unrepresentable.
- **Confidence gating** converts "automated and sometimes wrong" into "automated where safe, human where ambiguous". The threshold is tuned on labeled data, and the asymmetric cost of wrong actions is why the gate exists.
- **Redact before inference** — deterministic regexes strip keys, tokens, and emails locally; the model can double-check but must never be the primary redactor.
- Scale extraction in three gears: sequential (dozens) → async with a **semaphore cap** (thousands, politely) → **Batch APIs** (bulk jobs at ~50% price on all three providers, as of July 2026).
- Typed outputs make every project's result *ingestible* — queues, incident tools, and tables consume Pydantic objects; nobody writes a parser again.
- One helper (`extract()`) carried three different production patterns across three providers — that is what a good seam buys you.